# **IA explicable y reglas difusas: ¿Sinergia, redundancia, inutilidad o antagonismo?**

## **Subtítulo**

***Autor:*** Jorge Montesinos Bravo  
***Tutor:*** Mercedes Valdés Vela

***Fecha:*** 99/05/2026

Motivación: las reglas difusas han sido tradicionalmente modelos interpretables. Igual que la IA vivió su invierno, y ahora ha resurgido como el Avel Fénix, últimamente es el Soft computing y, concretamente la Lógica Fuzzy lo que han sido denostados por diferentes razones, entre otras que el deep learning lo ha fagocitado todo (gracias, a cantidades de deatos, potencia de cálculo, velocidad). Sin embarog tiene el problema de la explicabilidad: en este tfm se estudia la posiblidad de que las reglas difusas puedan encontrar su lugar en esta nueva elra de la IA aportando a la explicabilidad. Este viaje puede ser un fracaso por lo que el título del TFM es una pregunra: sinergia, redundancia o antagonismo?

CONTEXTUALIZACIÓN O ESTADO DEL ARTE

    CONTEXTUALIZACIÓN FUZZY
  
          SISTEMAS DE REGLAS DIFUSAS Y SOFT COMPUTING
        
                (fuente: mi tesis)
        
                EL INVIERNO DEL SOFT COMPUTING
        
                    básicamente el deep learning lo acapara todo
            
                        problema de estas técnicas: no explicabilidad
            
                CONTEXTUALIZACIÓN
    
            RESUMEN DE TODAS LAS TENDENCIAS ARA LLEGAR A ACOTAR LIME Y SHAP (model agnostic, local....)
        
                RESUMEN DE SHAP
        
                RESUMENT DE TODAS LAS COMBINACIONES DE LIME
        
                    confirmación de que no hay combinación con fuzzy (aunque sí con shap pero tendrá alguna pega que motiva nuestro trabajo)
            
                PROPUESTA
    
            COMBINAR LIME CON FUZZY:
        
                    COMBINACIÓN TIPO 1: ANFIS ES EL MODELO A EXPLICAR: LIME CON SU MODELO LINEAL ORIGINAL LO EXPLICA: ESTO VA A SER MALA IDEA
            
                        COMBINACIÓN TIPO 2: EL MODELO A EXPLICAR ES EL QUE SEA: FUZZY APORTA EL MODELO QUE EXPLICA
            
                        COMBINACIÓN TIPO 3:  EL MODELO A EXPLICAR ES EL QUE SEA: LIME CON SU MODELO LINEAL ORIGINAL Y SU RANKING DE VARIABLES PERO CON CON NÙMEROS
            DIFUSOS Y COMPARACION DIFUSA
            
              EXPERIMENTO
    
        EVALUACIÓN
    
        CONCLUSIONES

      Idealmente podría mostrarse algo así

                          SINERGIA     REDUNDANCIA    INUTILIDAD  ANTAGNISMO
                        
        COMBINACIÓN TIPO 1:    X
    
        COMBINACIÓN TIPO 2:                X
    
        COMBINACIÓN TIPO 3:    X  

## TODOs

* pasarle un modelo Random Forest, NN, etc y conseguir explicaciones tanto locales como globales (sacando estas a partir de muchas explicaciones locales, mirar el pdf de lime otra vez)

* ver la diferencia entre explicabilidad e interpretabilidad (sobre todo puede ayudar para la memoria)

* Interpretable Machine learning pdf

* Estado del arte de XAI, fuzzy y porque fuzzy está en "su invierno"

* ANFIS o cualquier herramienta que genera reglas difusas desde datos (implementacion/libreria)

## Librerías

In [1]:
import pandas as pd
import numpy as np
import random
import lime
from lime.lime_tabular import LimeTabularExplainer
from lime import lime_tabular, submodular_pick

from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.tree import export_text
import skfuzzy as fuzz
from simpful import *

import matplotlib.pyplot as plt
from tqdm import tqdm
import seaborn as sns

import IPython
from IPython.display import display, HTML
import sys

# Forzamos que IPython.core.display tenga el atributo display
IPython.core.display.display = IPython.display.display

import warnings
warnings.filterwarnings('ignore')

## Prueba de generación de variables

In [2]:
def generar_dataset_polinomico(n_instancias=1000, semilla=None):
    """
    Genera un dataset sintético con 6 variables y un target 
    basado en una función no lineal (cuadrática, trigonométrica y exponencial).
    Mantiene la estructura de retorno del dataset polinómico original.
    """
    if semilla is not None:
        np.random.seed(semilla)
        
    # 1. Generar los nombres de las columnas (x1, x2, x3, ...)
    columnas = [f'x{i}' for i in range(1, 7)]
    
    # 2. Generar la matriz de datos de entrada (features)
    # Mantenemos el rango [1, 3] de tu código original
    X = np.random.uniform(1, 3, size=(n_instancias, 6))
    
    # 3. Definir coeficientes y operaciones
    # Solo las 3 primeras variables tendrán impacto. El resto (0) serán ruido.
    coeficientes = [1, 10, -1, 0, 0, 0] 
    operaciones = ['x^2', 'sin(x)', 'exp(x/2)', 'ruido', 'ruido', 'ruido']
    
    # 4. Calcular la variable de salida (target) 'y' y construir la fórmula
    y = np.zeros(n_instancias)
    terminos_formula = []
    
    for i in range(6):
        if coeficientes[i] == 0:
            continue  # Si el coeficiente es 0, es ruido y no lo añadimos a la fórmula
            
        # Aplicamos la transformación no lineal según la operación definida
        if operaciones[i] == 'x^2':
            y += coeficientes[i] * (X[:, i] ** 2)
            termino_str = f"{coeficientes[i]}*{columnas[i]}^2"
            
        elif operaciones[i] == 'sin(x)':
            y += coeficientes[i] * np.sin(X[:, i])
            termino_str = f"{coeficientes[i]}*sin({columnas[i]})"
            
        elif operaciones[i] == 'exp(x/2)':
            y += coeficientes[i] * np.exp(X[:, i] / 2.0)
            termino_str = f"{coeficientes[i]}*exp({columnas[i]}/2)"
            
        terminos_formula.append(termino_str)
        
    # Limpiamos la fórmula generada (ej: "+ -1*" pasa a "- ") para que quede bonita al imprimirla
    formula_final = "y = " + " + ".join(terminos_formula).replace("+ -", "- ").replace("1*", "")
    
    # 5. Añadir un ligero ruido para que el modelo predictivo no memorice una función perfecta
    ruido = np.random.normal(0, 0.5, size=n_instancias)
    y += ruido
    
    # 6. Crear el DataFrame final
    df = pd.DataFrame(X, columns=columnas)
    df['target'] = y
    
    return df, formula_final, coeficientes, operaciones

In [3]:
# Vamos a generar un dataset con 6 variables
SEED = 1234
dataset, formula_real, coeficientes, operaciones = generar_dataset_polinomico(n_instancias=5000, semilla=SEED) # n_variables=6, 

np.random.seed(SEED)
random.seed(SEED)

# Imprimir la fórmula que el código se ha inventado para este dataset
print("La función real (Ground Truth) subyacente es:")
print(formula_real)
print("-" * 60)

# Mostrar el dataset resultante
display(dataset.head(20))

La función real (Ground Truth) subyacente es:
y = x1^2 + 10*sin(x2) - exp(x3/2)
------------------------------------------------------------


,x1,x2,x3,x4,x5,x6,target
0,1.383039,2.244218,1.875455,2.570717,2.559952,1.545185,7.456321
1,1.552929,2.603744,2.916279,2.751865,1.715635,2.001990,4.125893
2,2.366926,2.425404,1.740502,2.122392,2.006166,1.027537,9.488516
3,2.545653,2.765282,1.729772,2.230792,1.150762,1.737648,7.804366
4,2.866280,2.302756,1.794405,2.577460,1.633672,2.136197,13.206500
5,2.738255,1.872347,2.604295,1.287534,2.408522,2.409163,13.716595
6,1.437584,2.849735,1.884282,2.818632,1.119618,1.368574,2.033281
7,1.094711,2.349762,2.189250,2.066620,1.086648,2.122866,5.902504
8,1.659337,2.005934,1.223789,2.214387,2.131889,1.013528,10.431161
9,2.234883,2.824246,2.581048,2.984163,2.917604,2.583928,4.260023


In [4]:
max(dataset['target']), min(dataset['target'])

(17.71205927158771, -2.55427643846072)

## Division train test

In [5]:
# 1. Preparar los datos
X = dataset.drop('target', axis=1)
y = dataset['target']

# Dividir en entrenamiento y test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED)

## Random Forest

In [6]:
#grid search para encontrar el mejor modelo de random forest 
grid_rf = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

In [7]:
search_rf = GridSearchCV(
    estimator=RandomForestRegressor(random_state=SEED), 
    param_grid=grid_rf, 
    cv=5, 
    n_jobs=-1, 
    verbose=2
)
search_rf.fit(X_train, y_train)

Fitting 5 folds for each of 81 candidates, totalling 405 fits


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",RandomForestR...om_state=1234)
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'max_depth': [None, 10, ...], 'min_samples_leaf': [1, 2, ...], 'min_samples_split': [2, 5, ...], 'n_estimators': [50, 100, ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",None
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computation time for each fold and par

In [8]:
rf = search_rf.best_estimator_
y_pred = rf.predict(X_test)

## Explicador LIME

In [9]:
# 3. Inicializar el explicador LIME
# OJO para tu TFM: LIME por defecto discretiza las variables continuas en cuartiles.
# Para comparar matemáticamente con un polinomio continuo, a veces es mejor desactivarlo.
explicador = lime.lime_tabular.LimeTabularExplainer(
    training_data=np.array(X_train),
    feature_names=X_train.columns,
    mode='regression',
    discretize_continuous=False, # Fundamental para ver coeficientes lineales puros
    random_state=SEED
)

### Ejemplo de explicación

In [10]:
# 4. Elegir una instancia (fila) para explicar
indice_instancia = 0
instancia_a_explicar = X_test.iloc[indice_instancia]
valor_real = y_test.iloc[indice_instancia]
prediccion_modelo = y_pred[indice_instancia]

print(f"Explicando la instancia {indice_instancia} del conjunto de test:")
print(instancia_a_explicar.to_string())
print(f"\nValor Real (Polinomio): {valor_real:.4f}")
print(f"Predicción del Modelo: {prediccion_modelo:.4f}")
print("-" * 60)

Explicando la instancia 0 del conjunto de test:
x1    1.169868
x2    1.479261
x3    1.989312
x4    1.643785
x5    2.938514
x6    1.139461

Valor Real (Polinomio): 8.2937
Predicción del Modelo: 8.8270
------------------------------------------------------------


In [11]:

# # 5. Generar la explicación con LIME
# explicacion = explicador.explain_instance(
#     data_row=instancia_a_explicar,
#     predict_fn=rf.predict,
#     num_features=len(X_test.columns) # Que nos muestre el peso de todas las variables
# )

# # 6. Extraer y mostrar los pesos locales que LIME ha calculado
# print("Pesos locales calculados por LIME (Aproximación lineal local):")
# pesos_lime = explicacion.as_list()
# for feature, weight in pesos_lime:
#     print(f"Variable {feature}: {weight:.4f}")

# # Visualizar la explicación en el notebook
# explicacion.show_in_notebook(show_table=True)

# print("\nPolinomio real (Ground Truth) para esta instancia:")
# print(formula_real)

### Explicación global con SP-LIME

In [12]:
# # Configuración
# num_explicaciones_representativas = 5  # Cantidad de instancias a mostrar para entender el global
# num_instancias_a_muestrear = len(X_test) #100 # Muestra aleatoria de X_test a evaluar (mayor = mejor, pero más lento)

# # 1. Ejecutar el algoritmo Submodular Pick
# sp_explainer = submodular_pick.SubmodularPick(
#     explainer=explicador,
#     data=X_test.values,         # Es recomendable pasar los valores como array de numpy
#     predict_fn=rf.predict,      # Pasamos el modelo Random Forest
#     sample_size=num_instancias_a_muestrear,
#     num_features=len(X_test.columns),
#     num_exps_desired=num_explicaciones_representativas
# )

# # 2. Visualizar las explicaciones globales (el conjunto representativo)
# for i, exp in enumerate(sp_explainer.sp_explanations):
#     print(f"\n{'='*60}")
#     print(f"Explicación Global/Representativa {i+1}")
#     print(f"{'='*60}")
#     exp.show_in_notebook(show_table=True)

## Obtencion de los vecinos más cercanos de LIME mediante un wrapper para una predicción local

Comprobamos con un arbol de decisión si podría tener sentido cambiar la regresion lineal de LIME por reglas difusas:

Al imprimir estas reglas, verás algo parecido a:SI $X_4$ > 2.5 Y $X_3$ <= 1.2 ENTONCES Predicción = 540.2

Si compruebas que estas reglas rígidas capturan la no linealidad de tus potencias (grado 5) mucho mejor que los pesos lineales que te daba LIME, habrás validado técnicamente el Plan A. El siguiente reto será construir el motor difuso que suavice esas fronteras ($X_4$ > 2.5 se convertirá en $X_4$ es ALTO).

Una vez tengas estas reglas difusas locales funcionando para múltiples instancias, te enfrentarás al problema de cómo unirlas todas para crear el modelo global. Al analizar futuros grafos de selección para construir esa arquitectura global, considera que el algoritmo estándar de LIME (epsilon=0, puramente Greedy) tiene la tendencia a quedarse atascado en valores subóptimos debido a una falta de exploración.

Tener en tu poder el control absoluto de df_vecindario te da una ventaja inmensa: no tienes por qué depender de ese método de selección voraz. Puedes diseñar una estrategia de agregación difusa o un algoritmo genético que explore el espacio de soluciones de manera mucho más amplia, garantizando que el conjunto final de reglas difusas describa fielmente el comportamiento de tu Random Forest en todos los extremos de tu polinomio.

Ahora, pasemos a analizar la salida que has obtenido. ¡Es un resultado absolutamente brillante para tu tesis!

Fíjate en lo que acaba de hacer este simple árbol de decisión: ha ignorado por completo las variables $X_1, X_2, X_5$ y $X_6$. Al intentar explicar el vecindario local, el modelo basado en reglas ha deducido automáticamente que la predicción depende exclusiva y rotundamente de $X_3$ y $X_4$.

Si traducimos esas reglas matemáticas rígidas a lenguaje natural, el árbol ha descubierto exactamente el comportamiento de las potencias de grado 5 de tu polinomio:

La zona de máximo hundimiento (La penalización de $X_3$):

* Regla: |--- x3 > 2.52 y |--- x4 <= 2.39 y |--- x3 > 2.79 $\rightarrow$ value: [-581.74]
* Traducción: Si $X_3$ toma valores muy altos (cercanos a 3) y $X_4$ se mantiene bajo, la predicción se desploma a -581. El término $-3x_3^5$ domina por completo.

La zona de máxima explosión (El impulso de $X_4$):
* Regla: |--- x3 <= 2.52 y |--- x4 > 2.47 y |--- x4 > 2.76 $\rightarrow$ value: [546.80]
* Traducción: Si es $X_4$ quien toma valores muy altos y $X_3$ se mantiene bajo, la predicción se dispara a +546. El término $+3x_4^5$ toma el control.

La zona de conflicto (Interacciones):
* Regla: |--- x3 > 2.52 y |--- x4 > 2.39 y |--- x4 > 2.74 $\rightarrow$ value: [104.60]
* Traducción: ¿Qué pasa si ambas variables tienen valores altos simultáneamente? Al ser de signos opuestos, compiten entre sí. El árbol refleja que el resultado se modera muchísimo (104.60) porque se contrarrestan.

Esta prueba empírica demuestra que las reglas superan al modelo lineal clásico de LIME para representar interacciones complejas. Un modelo lineal habría dado un peso estático a $X_3$ y a $X_4$, pero nunca te habría explicado cómo interactúan entre ellas cuando ambas crecen.

## Bucle de obtención de reglas para N instancias diferentes

Realizamos este bucle de modo que se puedan ver qué patrones extrae el sistema en diferentes zonas del polinomio.

El objetivo de este primer bloque de código es automatizar lo que hiciste a mano en el paso anterior, pero para múltiples instancias simultáneamente, y recolectar los "puntos de corte" (los valores numéricos donde el árbol decide separar las ramas) para las variables críticas.Al considerar que el algoritmo de selección interno que propone LIME para representaciones globales (el método voraz con $\epsilon=0$) tiende a estancarse en valores subóptimos por falta de exploración, en esta recolección masiva vamos a iterar directamente sobre una muestra diversa de tu conjunto de test. Esto nos garantiza capturar la variabilidad real del polinomio en todo su dominio, evitando que los puntos de corte se sesguen hacia una sola región.Añade la siguiente celda a tu cuaderno. Este código utiliza la misma "función espía" que creamos antes, pero ahora se sumerge en la estructura interna del árbol de decisión de Scikit-Learn para extraer matemáticamente los umbrales numéricos de las reglas.

### ANOTACIÓN IMPORTANTE

Usamos lime solamente para generar las reglas pero no hacemos caso de sus predicciones. Entonces el proceso de "generación de reglas" y su aplicación se podrían sustituir en un futuro en lugar del regresor lineal de lime

Explicación de Gemini:

Si en un futuro decidieras publicar tu propia librería en Python (por ejemplo, haciendo un fork del repositorio oficial de LIME en GitHub), literalmente irías a su código fuente, borrarías las líneas donde entrenan el modelo lineal, e insertarías la llamada a tu generador de reglas difusas. De cara al usuario final, la herramienta funcionaría igual: le pasas una instancia y te devuelve una explicación, pero en lugar de un gráfico de barras confuso con pendientes locales, le devolvería un conjunto de reglas lingüísticas claras.

Aquí tienes la hoja de ruta corregida y ordenada, con cada paso resumido en una sola frase tal y como pedías:

- Fuzzificación del dominio: Definir matemáticamente las funciones de pertenencia (MUY ALTO, ALTO, MEDIO, BAJO, MUY BAJO) a partir de los histogramas para establecer el vocabulario difuso del sistema.

- Extracción y traducción masiva: Extraer las reglas rígidas locales de las perturbaciones generadas por LIME para todo el conjunto de test y traducirlas automáticamente al vocabulario difuso.

- Agregación global: Agrupar estas miles de reglas locales en un conjunto representativo mediante técnicas avanzadas, evitando que el clásico algoritmo de optimización puramente voraz con epsilon=0 se quede atascado en valores subóptimos por su falta de exploración.

- Construcción del modelo: Diseñar el Sistema de Inferencia Difuso (FIS) alimentado por esas reglas globales para que sea capaz de predecir valores y emitir explicaciones lingüísticas.

- Encapsulamiento de la herramienta: Modificar el código para sustituir completamente la regresión lineal de LIME por nuestro nuevo método de inferencia difuso, logrando un explicador totalmente agnóstico.

## Fuzzificación de variables encontradas en las reglas

### Supuesto pipeline a partir de ahora (obtencion de reglas rigidas, fuzzificacion de las variables, fuzzificacion de las reglas, etc.)

#### Configuración y variables globales

In [13]:
# ==========================================
# CONFIGURACIÓN Y VARIABLES GLOBALES
# ==========================================
datos_perturbados_lime = None
predicciones_rf_lime = None

def funcion_espia(X):
    global datos_perturbados_lime, predicciones_rf_lime
    datos_perturbados_lime = X 
    preds = rf.predict(X)
    predicciones_rf_lime = preds
    return preds

# Estructuras de almacenamiento global
dict_cortes = {}             # Para calcular los percentiles
lista_reglas_crudas = []     # El "saco" de reglas rígidas

# Función auxiliar para extraer caminos de decisión estructurados del árbol
def extraer_reglas_arbol(dt, nombres_variables, id_instancia):
    tree_ = dt.tree_
    
    def recorrer(nodo, camino_actual):
        # Si es un nodo hoja, guardamos la regla completa
        if tree_.children_left[nodo] == tree_.children_right[nodo]:
            prediccion = tree_.value[nodo][0][0]
            lista_reglas_crudas.append({
                'instancia_origen': id_instancia,
                'condiciones': camino_actual,
                'prediccion': prediccion
            })
            return
        
        # Si es un nodo de decisión, extraemos el corte
        feature_name = nombres_variables[tree_.feature[nodo]]
        threshold = tree_.threshold[nodo]
        
        # Alimentamos el diccionario dinámico de cortes
        if feature_name not in dict_cortes:
            dict_cortes[feature_name] = []
        dict_cortes[feature_name].append(threshold)
        
        # Explorar rama izquierda (<=) y derecha (>)
        recorrer(tree_.children_left[nodo], camino_actual + [(feature_name, '<=', threshold)])
        recorrer(tree_.children_right[nodo], camino_actual + [(feature_name, '>', threshold)])
        
    recorrer(0, []) # Iniciar desde la raíz

#### Fase 1: Extracción Masiva de reglas

In [14]:
# ==========================================
# FASE 1: EXTRACCIÓN MASIVA
# ==========================================
num_instancias_a_procesar = len(X_train) # Cambiar a len(X_train) para ejecución final
print(f"--- FASE 1: Extrayendo vecindarios y reglas de {num_instancias_a_procesar} instancias ---")

for i in range(num_instancias_a_procesar):
    instancia = X_train.iloc[i]
    
    # 1. Generar vecindario con LIME (usando la función espía)
    _ = explicador.explain_instance(
        data_row=instancia,
        predict_fn=funcion_espia,
        num_features=len(X_train.columns)
    )
    
    # 2. Reconstruir datos locales
    df_vecindario = pd.DataFrame(datos_perturbados_lime, columns=X_train.columns)
    
    # 3. Entrenar el traductor local (Árbol rígido)
    dt = DecisionTreeRegressor(max_depth=3, random_state=SEED) # Profundidad 2 para reglas simples pero probar con 3 o 4 para reglas más reales aunque legibles.
    dt.fit(df_vecindario, predicciones_rf_lime)
    
    # 4. Extraer cortes y reglas al mismo tiempo
    extraer_reglas_arbol(dt, X_train.columns, id_instancia=i)

print(f"Extracción completada. Se generaron {len(lista_reglas_crudas)} reglas rígidas locales.\n")

--- FASE 1: Extrayendo vecindarios y reglas de 4000 instancias ---
Extracción completada. Se generaron 32000 reglas rígidas locales.



#### Fase 2: Fuzzyficación Dinámica (Vocabulario)

In [15]:
# ==============================================
# FASE 2: FUZZIFICACIÓN DINÁMICA (VOCABULARIO)
# ==============================================
print("--- FASE 2: Construyendo Vocabulario Difuso Universal ---")
dict_vocabulario_difuso = {}

for feature, cortes in dict_cortes.items():
    min_val = X_train[feature].min()
    max_val = X_train[feature].max()
    universo = np.linspace(min_val, max_val, 200)
    
    p20, p40, p60, p80 = np.percentile(cortes, [20, 40, 60, 80])
    
    dict_vocabulario_difuso[feature] = {
        'universo': universo,
        'MUY BAJO': fuzz.trapmf(universo, [min_val, min_val, p20, p40]),
        'BAJO': fuzz.trimf(universo, [p20, p40, p60]),
        'MEDIO': fuzz.trimf(universo, [p40, p60, p80]),
        'ALTO': fuzz.trimf(universo, [p60, p80, max_val]),
        'MUY ALTO': fuzz.trapmf(universo, [p80, max_val, max_val, max_val])
    }
    
print("Vocabulario construido basándose en la estadística global de los cortes.\n")


print("--- FASE 2.1: Fuzzificación Dinámica de la Variable de Salida (Predicción) ---")

# 1. Recopilamos todas las predicciones crudas que sacamos en la Fase 1
todas_las_predicciones = [regla['prediccion'] for regla in lista_reglas_crudas]

# 2. Calculamos sus percentiles dinámicos
min_pred = min(todas_las_predicciones)
max_pred = max(todas_las_predicciones)
universo_pred = np.linspace(min_pred, max_pred, 200)

p20, p40, p60, p80 = np.percentile(todas_las_predicciones, [20, 40, 60, 80])

vocabulario_salida = {
    'universo': universo_pred,
    'MUY BAJA': fuzz.trapmf(universo_pred, [min_pred, min_pred, p20, p40]),
    'BAJA': fuzz.trimf(universo_pred, [p20, p40, p60]),
    'MEDIA': fuzz.trimf(universo_pred, [p40, p60, p80]),
    'ALTA': fuzz.trimf(universo_pred, [p60, p80, max_pred]),
    'MUY ALTA': fuzz.trapmf(universo_pred, [p80, max_pred, max_pred, max_pred])
}
print("Vocabulario de salida construido con éxito.\n")

--- FASE 2: Construyendo Vocabulario Difuso Universal ---
Vocabulario construido basándose en la estadística global de los cortes.

--- FASE 2.1: Fuzzificación Dinámica de la Variable de Salida (Predicción) ---
Vocabulario de salida construido con éxito.



#### Fase 3: Traductor Lingüístico.

Si las reglas del arbol de decisión han generado reglas del tipo `Si x3>= 0.5 Y x4>=1 Y x3<1.5` ahora no las traduciremos como `Si x3 es MEDIO Y x4 es ALTO Y x3 es MUY ALTO` sino que como x3 tiene un límite tanto superior como inferior, obtendremos el punto medio de esos límites y quedará una regla parecida a `Si x3 es ALTO Y x4 es ALTO` suponiendo que el punto medio de esos límites cae en el intervalo de `ALTO`.

In [16]:
# ==========================================
# FASE 3: TRADUCTOR LINGÜÍSTICO EN MEMORIA
# ==========================================
import numpy as np
import pandas as pd

print("--- FASE 3 : Traducción Lingüística con Consolidación de Intervalos ---")
lista_reglas_difusas = []

for regla in lista_reglas_crudas:
    condiciones_difusas = []
    
    # 1. Consolidar la lógica del árbol en intervalos numéricos puros
    intervalos = {}
    for (feature, operador, umbral) in regla['condiciones']:
        if feature not in intervalos:
            # Inicializamos con el mínimo y máximo global del universo de esa variable
            vocab = dict_vocabulario_difuso[feature]
            min_global = vocab['universo'][0]
            max_global = vocab['universo'][-1]
            intervalos[feature] = {'min': min_global, 'max': max_global}
        
        # Acotamos el intervalo según la decisión del árbol
        if operador == '<=':
            intervalos[feature]['max'] = min(intervalos[feature]['max'], umbral)
        elif operador == '>':
            intervalos[feature]['min'] = max(intervalos[feature]['min'], umbral)
            
    # 2. Calcular el punto representativo (punto medio) y Fuzzificar
    for feature, limites in intervalos.items():
        punto_medio = (limites['min'] + limites['max']) / 2.0
        
        vocab = dict_vocabulario_difuso[feature]
        idx_umbral = (np.abs(vocab['universo'] - punto_medio)).argmin()
        
        pertenencia = {etiq: vocab[etiq][idx_umbral] for etiq in vocab.keys() if etiq != 'universo'}
        etiqueta_ganadora = max(pertenencia, key=pertenencia.get)
        
        condiciones_difusas.append(f"{feature} es {etiqueta_ganadora}")
        
    # Ordenamos alfabéticamente para mantener la simetría lógica
    condicion_ordenada = " Y ".join(sorted(condiciones_difusas))
    
    # --- Parte 3: Traducir el consecuente (Predicción) ---
    pred = regla['prediccion']
    idx_pred = (np.abs(vocabulario_salida['universo'] - pred)).argmin()
    
    pertenencia_pred = {etiq: vocabulario_salida[etiq][idx_pred] for etiq in vocabulario_salida.keys() if etiq != 'universo'}
    etiqueta_pred_ganadora = max(pertenencia_pred, key=pertenencia_pred.get)
    
    # Unimos todo y guardamos
    lista_reglas_difusas.append({
        'Condicion': condicion_ordenada,
        'Prediccion_Linguistica': etiqueta_pred_ganadora
    })

print("¡Traducción completada! Muestra aleatoria del banco global de reglas difusas:")
for i in range(min(5, len(lista_reglas_difusas))):
    ejemplo = random.choice(lista_reglas_difusas)
    print(f"SI {ejemplo['Condicion']} ENTONCES Predicción = {ejemplo['Prediccion_Linguistica']}")

--- FASE 3 : Traducción Lingüística con Consolidación de Intervalos ---
¡Traducción completada! Muestra aleatoria del banco global de reglas difusas:
SI x1 es MUY BAJO Y x2 es BAJO ENTONCES Predicción = MUY BAJA
SI x1 es ALTO Y x2 es MUY BAJO ENTONCES Predicción = ALTA
SI x1 es BAJO Y x2 es MUY BAJO ENTONCES Predicción = MEDIA
SI x1 es MUY BAJO Y x2 es BAJO ENTONCES Predicción = MUY BAJA
SI x1 es MUY BAJO Y x2 es ALTO ENTONCES Predicción = MUY BAJA


#### Fase 4: Agregación Global y Resolución de Conflictos

In [17]:
print("--- FASE 4 : Agregación Global y Resolución de Conflictos ---")
df_reglas = pd.DataFrame(lista_reglas_difusas)

# 1. Contamos las frecuencias de cada combinación exacta (Condición + Predicción)
df_frecuencias = df_reglas.groupby(['Condicion', 'Prediccion_Linguistica']).size().reset_index(name='Frecuencia')

# 2. RESOLUCIÓN DE CONFLICTOS: Para cada Condición, nos quedamos con la fila de mayor Frecuencia
# Ordenamos por Frecuencia descendente y luego eliminamos duplicados en 'Condicion' manteniendo el primero (el mayor)
df_global_sin_conflictos = df_frecuencias.sort_values(by='Frecuencia', ascending=False).drop_duplicates(subset=['Condicion'], keep='first')

# 3. Ordenamos el resultado final para verlo de forma limpia
df_global_sin_conflictos = df_global_sin_conflictos.sort_values(by='Frecuencia', ascending=False).reset_index(drop=True)

print(f"Banco final purificado: {len(df_global_sin_conflictos)} reglas lingüísticas maestras.\n") #  (Máximo teórico: 125)
display(df_global_sin_conflictos.head(25))

--- FASE 4 : Agregación Global y Resolución de Conflictos ---
Banco final purificado: 15 reglas lingüísticas maestras.



,Condicion,Prediccion_Linguistica,Frecuencia
0,x1 es MUY BAJO Y x2 es MUY BAJO,BAJA,4000
1,x1 es MUY BAJO Y x2 es BAJO,MUY BAJA,3398
2,x1 es BAJO Y x2 es MUY BAJO,MEDIA,3354
3,x1 es MUY BAJO Y x2 es ALTO,MUY BAJA,3346
4,x1 es MUY ALTO Y x2 es MUY BAJO,MUY ALTA,3238
5,x1 es ALTO Y x2 es BAJO,MEDIA,2751
6,x1 es ALTO Y x2 es ALTO,MUY BAJA,2557
7,x1 es ALTO Y x2 es MUY BAJO,ALTA,2482
8,x1 es MEDIO Y x2 es MUY BAJO,ALTA,1518
9,x1 es ALTO Y x2 es MUY ALTO,MUY BAJA,1388


#### Fase 5: Fuzzy Clustering

In [18]:
print("--- FASE 5: Poda por Soporte para Enfoque Aproximativo de Mamdani ---")

# Definimos el soporte mínimo (ej. 0.05 significa que la regla debe explicar al menos al 5% de los datos). 
# Las reglas que caigan por debajo de este umbral se descartan como "ruido" o sobreajuste de LIME.
MIN_SOPORTE = 0.02  # <-- Ajusta este umbral para ser más estricto o más permisivo

# 1. Calculamos el "Universo de Explicaciones" (Total de instancias que suman todas nuestras reglas)
total_instancias = df_global_sin_conflictos['Frecuencia'].sum()
print(f"Total de instancias cubiertas por las reglas base: {total_instancias}")

# 2. Calculamos el Soporte (peso real) de cada regla en el modelo
df_mamdani = df_global_sin_conflictos.copy()
df_mamdani['Soporte'] = df_mamdani['Frecuencia'] / total_instancias

# 3. Aplicamos la Poda Mamdani (Filtramos las reglas robustas)
df_modelo_final = df_mamdani[df_mamdani['Soporte'] >= MIN_SOPORTE].copy()

# 4. Ordenamos de mayor a menor importancia para facilitar la lectura humana
df_modelo_final = df_modelo_final.sort_values(by='Frecuencia', ascending=False).reset_index(drop=True)

# 5. Formateamos visualmente para la tabla de salida
df_modelo_final['Soporte_Porcentaje'] = (df_modelo_final['Soporte'] * 100).apply(lambda x: f"{x:.2f}%")

print(f"\n¡Poda completada con éxito!")
print(f"Reglas originales extraídas de LIME: {len(df_global_sin_conflictos)}")
print(f"Reglas supervivientes para el motor Mamdani: {len(df_modelo_final)}\n")

# Mostramos el diccionario lingüístico final que alimentará la inferencia
display(df_modelo_final) #[['Condicion', 'Prediccion_Linguistica', 'Frecuencia', 'Soporte_Porcentaje']]

--- FASE 5: Poda por Soporte para Enfoque Aproximativo de Mamdani ---
Total de instancias cubiertas por las reglas base: 28698

¡Poda completada con éxito!
Reglas originales extraídas de LIME: 15
Reglas supervivientes para el motor Mamdani: 11



,Condicion,Prediccion_Linguistica,Frecuencia,Soporte,Soporte_Porcentaje
0,x1 es MUY BAJO Y x2 es MUY BAJO,BAJA,4000,0.139383,13.94%
1,x1 es MUY BAJO Y x2 es BAJO,MUY BAJA,3398,0.118405,11.84%
2,x1 es BAJO Y x2 es MUY BAJO,MEDIA,3354,0.116872,11.69%
3,x1 es MUY BAJO Y x2 es ALTO,MUY BAJA,3346,0.116593,11.66%
4,x1 es MUY ALTO Y x2 es MUY BAJO,MUY ALTA,3238,0.112830,11.28%
5,x1 es ALTO Y x2 es BAJO,MEDIA,2751,0.095860,9.59%
6,x1 es ALTO Y x2 es ALTO,MUY BAJA,2557,0.089100,8.91%
7,x1 es ALTO Y x2 es MUY BAJO,ALTA,2482,0.086487,8.65%
8,x1 es MEDIO Y x2 es MUY BAJO,ALTA,1518,0.052896,5.29%
9,x1 es ALTO Y x2 es MUY ALTO,MUY BAJA,1388,0.048366,4.84%


#### Predicción con reglas difusas (Motor de Inferencia Difuso de Mamdani)

In [19]:
def predecir_y_explicar_con_reglas(instancia, df_reglas, vocab_entrada, vocab_salida):
    # 1. FUZZIFICACIÓN DE ENTRADAS
    pertenencias_entrada = {}
    for feature in vocab_entrada.keys():
        val = instancia[feature]
        vocab = vocab_entrada[feature]
        universo = vocab['universo']
        
        val_clipped = np.clip(val, universo[0], universo[-1])
        idx = (np.abs(universo - val_clipped)).argmin()
        
        pertenencias_entrada[feature] = {etiq: vocab[etiq][idx] for etiq in vocab.keys() if etiq != 'universo'}
                
    # 2. EVALUACIÓN DE REGLAS Y EXTRACCIÓN DE EXPLICACIONES
    activaciones_salida = {etiq: 0.0 for etiq in vocab_salida.keys() if etiq != 'universo'}
    reglas_disparadas = [] # ¡Aquí guardaremos las explicaciones!
    
    for _, row in df_reglas.iterrows():
        condicion = row['Condicion']
        pred_ling = row['Prediccion_Linguistica']
        
        partes = condicion.split(" Y ")
        grados_condicion = []
        
        for parte in partes:
            var, _, etiq = parte.partition(" es ")
            if var in pertenencias_entrada and etiq in pertenencias_entrada[var]:
                grados_condicion.append(pertenencias_entrada[var][etiq])
            else:
                grados_condicion.append(0.0)
                
        grado_activacion = min(grados_condicion) if grados_condicion else 0.0
        
        # Si la regla se ha activado (su fuerza es mayor a 0)
        if grado_activacion > 0:
            activaciones_salida[pred_ling] = max(activaciones_salida[pred_ling], grado_activacion)
            # Guardamos la regla como parte de la explicación
            reglas_disparadas.append({
                'Fuerza_Activacion': grado_activacion,
                'Texto_Regla': f"SI {condicion} ENTONCES Prediccion = {pred_ling}"
            })
            
    # Ordenamos las explicaciones para que las reglas más fuertes salgan primero
    reglas_disparadas = sorted(reglas_disparadas, key=lambda x: x['Fuerza_Activacion'], reverse=True)
        
    # 3. SALIDA LINGÜÍSTICA (Para la tutora y el usuario)
    if sum(activaciones_salida.values()) == 0:
        pred_linguistica_oficial = "DESCONOCIDA (Falta cobertura)"
        pred_crisp_oculta = np.mean(vocab_salida['universo'])
    else:
        # La etiqueta ganadora es la que más fuerza haya acumulado
        pred_linguistica_oficial = max(activaciones_salida, key=activaciones_salida.get)
        
        # 4. SALIDA NUMÉRICA OCULTA (Solo para calcular el Error frente al Random Forest)
        universo_out = vocab_salida['universo']
        agregacion_final = np.zeros_like(universo_out)
        for etiq, activacion in activaciones_salida.items():
            if activacion > 0:
                agregacion_final = np.fmax(agregacion_final, np.fmin(activacion, vocab_salida[etiq]))
        pred_crisp_oculta = fuzz.defuzz(universo_out, agregacion_final, 'centroid')
        
    return {
        'Prediccion_Linguistica': pred_linguistica_oficial,
        'Prediccion_Numerica': pred_crisp_oculta,
        'Explicaciones': reglas_disparadas
    }

In [20]:
# ===============================================
# TESTEO DEL MODELO CON INTERFAZ DE EXPLICACIÓN
# ===============================================

reglas_a_usar = df_modelo_final # df_global_sin_conflictos # Usa tu DataFrame purificado

# Vamos a probar solo con 3 instancias para leer bien las explicaciones
for i in range(3):
    instancia_test = X_test.iloc[i]
    pred_rf = rf.predict([instancia_test])[0]
    
    paquete_explicacion = predecir_y_explicar_con_reglas(
        instancia_test, reglas_a_usar, dict_vocabulario_difuso, vocabulario_salida
    )
    
    print(f"==================================================")
    print(f" INSTANCIA DE TEST #{i}")
    print(f"==================================================")
    print(f"Predicción original Caja Negra (RF): {pred_rf:.2f}")
    print(f"Predicción oficial Fuzzy-LIME:       >>> {paquete_explicacion['Prediccion_Linguistica']} <<<")
    print(f"   *(Valor numérico inferido: {paquete_explicacion['Prediccion_Numerica']:.2f})*")
    print(f"\n ¿POR QUÉ? (Explicación del Modelo):")
    
    if not paquete_explicacion['Explicaciones']:
        print("  Ninguna regla del modelo global cubre esta instancia.")
    else:
        for exp in paquete_explicacion['Explicaciones']:
            fuerza = exp['Fuerza_Activacion']
            texto = exp['Texto_Regla']
            # Mostramos como porcentaje de fuerza para que sea intuitivo
            print(f"  [Fuerza: {fuerza*100:05.2f}%] -> {texto}")
    print("\n")

 INSTANCIA DE TEST #0
Predicción original Caja Negra (RF): 8.83
Predicción oficial Fuzzy-LIME:       >>> BAJA <<<
   *(Valor numérico inferido: 7.49)*

 ¿POR QUÉ? (Explicación del Modelo):
  [Fuerza: 100.00%] -> SI x1 es MUY BAJO Y x2 es MUY BAJO ENTONCES Prediccion = BAJA


 INSTANCIA DE TEST #1
Predicción original Caja Negra (RF): 12.11
Predicción oficial Fuzzy-LIME:       >>> ALTA <<<
   *(Valor numérico inferido: 12.08)*

 ¿POR QUÉ? (Explicación del Modelo):
  [Fuerza: 58.68%] -> SI x1 es ALTO Y x2 es MUY BAJO ENTONCES Prediccion = ALTA
  [Fuerza: 41.32%] -> SI x1 es MUY ALTO Y x2 es MUY BAJO ENTONCES Prediccion = MUY ALTA


 INSTANCIA DE TEST #2
Predicción original Caja Negra (RF): 13.11
Predicción oficial Fuzzy-LIME:       >>> ALTA <<<
   *(Valor numérico inferido: 12.11)*

 ¿POR QUÉ? (Explicación del Modelo):
  [Fuerza: 56.23%] -> SI x1 es ALTO Y x2 es MUY BAJO ENTONCES Prediccion = ALTA
  [Fuerza: 43.77%] -> SI x1 es MUY ALTO Y x2 es MUY BAJO ENTONCES Prediccion = MUY ALTA




## Creación de la API

In [21]:
class ExplicacionDifusa:
    """Objeto devuelto por el explicador con la interfaz visual interactiva."""
    def __init__(self, paquete, valor_real=None):
        self.pred_ling = paquete['Prediccion_Linguistica']
        self.pred_num = paquete['Prediccion_Numerica']
        self.explicaciones = paquete['Explicaciones']
        self.valor_real = valor_real

    def show_in_notebook(self):
        html = f"""
        <div style="font-family: Arial, sans-serif; border: 2px solid #4CAF50; border-radius: 10px; padding: 15px; max-width: 650px; background-color: #f9f9f9; box-shadow: 2px 2px 10px rgba(0,0,0,0.1);">
            <h3 style="color: #4CAF50; margin-top: 0;">🧠 Explicación Fuzzy-LIME</h3>
        """
        if self.valor_real is not None:
             html += f"<p><b>Predicción de la Caja Negra:</b> {self.valor_real:.2f}</p>"
             
        html += f"""
            <p style="font-size: 1.1em;"><b>Diagnóstico Oficial:</b> <span style="background-color: #4CAF50; color: white; padding: 4px 10px; border-radius: 5px; font-weight: bold;">{self.pred_ling}</span></p>
            <p style="color: #666; font-size: 0.9em;">(Valor numérico inferido del centroide: {self.pred_num:.2f})</p>
            <hr style="border: 1px solid #ddd;">
            <h4 style="color: #333;">Desglose de Reglas Activadas:</h4>
            <ul style="list-style-type: none; padding-left: 0;">
        """
        if not self.explicaciones:
            html += "<li style='color: red;'>Ninguna regla del modelo global cubre esta instancia concreta.</li>"
        else:
            for exp in self.explicaciones:
                fuerza = exp['Fuerza_Activacion'] * 100
                color = "#4CAF50" if fuerza > 50 else "#FFC107" if fuerza > 20 else "#F44336"
                html += f"""
                <li style="margin-bottom: 15px;">
                    <div style="background-color: #e0e0e0; border-radius: 5px; height: 10px; margin-bottom: 5px;">
                        <div style="background-color: {color}; width: {fuerza}%; height: 100%; border-radius: 5px;"></div>
                    </div>
                    <b>[{fuerza:05.2f}%]</b> {exp['Texto_Regla']}
                </li>
                """
        html += "</ul></div>"
        display(HTML(html))

In [22]:
class FuzzyLimeExplainer:
    """
    Explicador Global basado en Lógica Difusa.
    Encapsula extracción, traducción, clustering e inferencia Mamdani.
    """
    def __init__(self, predict_fn, training_data, feature_names=None, mode='regression', 
                 max_depth=3, min_soporte=0, num_instancias_a_procesar=None, random_state=1234):
        """
        Parámetros:
        - predict_fn: Función de predicción del modelo caja negra (ej: rf.predict).
        - training_data: DataFrame o Numpy Array con los datos de entrenamiento.
        - feature_names: Lista de nombres de las variables (si training_data es DataFrame, se pueden inferir, si no, se generan nombres por defecto).
        - mode: 'regression' o 'classification' (LIME lo necesita para saber cómo generar el vecindario). En nuestro caso de momento solo hemos implementado para regresión, pero se podría adaptar a clasificación.
        - max_depth: Profundidad máxima para los árboles rígidos locales (fase de extracción).
        - min_soporte: Soporte mínimo para que una regla sea considerada en el modelo final.
        - num_instancias_a_procesar: Cuántas instancias del training_data usar para la extracción masiva (fase 1). Si None, se usan todas.
        """
        # GESTIÓN INTELIGENTE DE DATOS Y NOMBRES DE VARIABLES
        if isinstance(training_data, pd.DataFrame):
            self.training_data = training_data
            # Si no pasan nombres, cogemos las columnas del DataFrame
            self.feature_names = feature_names if feature_names is not None else list(training_data.columns)
        else:
            # Si pasan un Numpy Array, generamos nombres por defecto si no los dan
            self.feature_names = feature_names if feature_names is not None else [f"feature_{i}" for i in range(training_data.shape[1])]
            self.training_data = pd.DataFrame(training_data, columns=self.feature_names)

        self.predict_fn = predict_fn
        self.mode = mode
        
        self.max_depth = max_depth
        self.min_soporte = min_soporte
        self.num_instancias = num_instancias_a_procesar if num_instancias_a_procesar else len(training_data)
        self.random_state = random_state


        self.dict_cortes = {}
        self.lista_reglas_crudas = []
        self.vocabulario_entrada = {}
        self.vocabulario_salida = {}
        self.modelo_global_difuso = None
        
        self._temp_X = None
        self._temp_preds = None

    def _funcion_espia_interna(self, X):
        self._temp_X = X
        preds = self.predict_fn(X)
        self._temp_preds = preds
        return preds

    def _extraer_reglas_arbol(self, dt, feature_names):
        tree_ = dt.tree_
        def recorrer(nodo, camino_actual):
            if tree_.children_left[nodo] == tree_.children_right[nodo]:
                self.lista_reglas_crudas.append({
                    'condiciones': camino_actual,
                    'prediccion': tree_.value[nodo][0][0]
                })
                return
            feature_name = feature_names[tree_.feature[nodo]]
            threshold = tree_.threshold[nodo]
            
            if feature_name not in self.dict_cortes:
                self.dict_cortes[feature_name] = []
            self.dict_cortes[feature_name].append(threshold)
            
            recorrer(tree_.children_left[nodo], camino_actual + [(feature_name, '<=', threshold)])
            recorrer(tree_.children_right[nodo], camino_actual + [(feature_name, '>', threshold)])
        recorrer(0, [])

    def fit(self):
        """Ejecuta todo el pipeline (Fases 1 a 5)"""
        print(f"Iniciando entrenamiento de Fuzzy-LIME ({self.num_instancias} instancias)...")
        np.random.seed(self.random_state)
        
        # --- FASE 1: Extracción ---
        lime_base = LimeTabularExplainer(self.training_data.values, feature_names=self.feature_names, mode=self.mode, discretize_continuous=False, random_state=self.random_state)
        
        for i in range(self.num_instancias):
            instancia = self.training_data.iloc[i]
            _ = lime_base.explain_instance(instancia, self._funcion_espia_interna, num_features=len(self.feature_names))
            
            df_vecindario = pd.DataFrame(self._temp_X, columns=self.feature_names)
            dt = DecisionTreeRegressor(max_depth=self.max_depth, random_state=self.random_state)
            dt.fit(df_vecindario, self._temp_preds)
            self._extraer_reglas_arbol(dt, self.feature_names)

        # --- FASE 2: Vocabularios ---
        for feature, cortes in self.dict_cortes.items():
            min_val, max_val = self.training_data[feature].min(), self.training_data[feature].max()
            universo = np.linspace(min_val, max_val, 200)
            p20, p40, p60, p80 = np.percentile(cortes, [20, 40, 60, 80])
            self.vocabulario_entrada[feature] = {
                'universo': universo,
                'MUY BAJO': fuzz.trapmf(universo, [min_val, min_val, p20, p40]),
                'BAJO': fuzz.trimf(universo, [p20, p40, p60]),
                'MEDIO': fuzz.trimf(universo, [p40, p60, p80]),
                'ALTO': fuzz.trimf(universo, [p60, p80, max_val]),
                'MUY ALTO': fuzz.trapmf(universo, [p80, max_val, max_val, max_val])
            }
            
        todas_preds = [r['prediccion'] for r in self.lista_reglas_crudas]
        min_p, max_p = min(todas_preds), max(todas_preds)
        universo_p = np.linspace(min_p, max_p, 200)
        p20, p40, p60, p80 = np.percentile(todas_preds, [20, 40, 60, 80])
        self.vocabulario_salida = {
            'universo': universo_p,
            'MUY BAJA': fuzz.trapmf(universo_p, [min_p, min_p, p20, p40]),
            'BAJA': fuzz.trimf(universo_p, [p20, p40, p60]),
            'MEDIA': fuzz.trimf(universo_p, [p40, p60, p80]),
            'ALTA': fuzz.trimf(universo_p, [p60, p80, max_p]),
            'MUY ALTA': fuzz.trapmf(universo_p, [p80, max_p, max_p, max_p])
        }

        # --- FASE 3: Traducción con Consolidación de Intervalos ---
        lista_reglas_difusas_completas = []
        for regla in self.lista_reglas_crudas:
            intervalos = {}
            for (feature, op, umbral) in regla['condiciones']:
                if feature not in intervalos:
                    vocab = self.vocabulario_entrada[feature]
                    intervalos[feature] = {'min': vocab['universo'][0], 'max': vocab['universo'][-1]}
                if op == '<=':
                    intervalos[feature]['max'] = min(intervalos[feature]['max'], umbral)
                elif op == '>':
                    intervalos[feature]['min'] = max(intervalos[feature]['min'], umbral)
                    
            condiciones_difusas = []
            for feature, limites in intervalos.items():
                punto_medio = (limites['min'] + limites['max']) / 2.0
                vocab = self.vocabulario_entrada[feature]
                idx_umbral = (np.abs(vocab['universo'] - punto_medio)).argmin()
                pertenencia = {etiq: vocab[etiq][idx_umbral] for etiq in vocab.keys() if etiq != 'universo'}
                etiqueta_ganadora = max(pertenencia, key=pertenencia.get)
                condiciones_difusas.append(f"{feature} es {etiqueta_ganadora}")
                
            condicion_ordenada = " Y ".join(sorted(condiciones_difusas))
            pred = regla['prediccion']
            idx_pred = (np.abs(self.vocabulario_salida['universo'] - pred)).argmin()
            pertenencia_pred = {etiq: self.vocabulario_salida[etiq][idx_pred] for etiq in self.vocabulario_salida.keys() if etiq != 'universo'}
            etiqueta_pred_ganadora = max(pertenencia_pred, key=pertenencia_pred.get)
            
            lista_reglas_difusas_completas.append({
                'Condicion': condicion_ordenada,
                'Prediccion_Linguistica': etiqueta_pred_ganadora
            })

        # --- FASE 4: Resolución de Conflictos ---
        df_reglas = pd.DataFrame(lista_reglas_difusas_completas)
        df_frecuencias = df_reglas.groupby(['Condicion', 'Prediccion_Linguistica']).size().reset_index(name='Frecuencia')
        df_global_sin_conflictos = df_frecuencias.sort_values(by='Frecuencia', ascending=False).drop_duplicates(subset=['Condicion'], keep='first')

        # --- FASE 5: Fuzzy C-Means Clustering ---
        if self.min_soporte is not 0:
            total_instancias = df_global_sin_conflictos['Frecuencia'].sum()
            print(f"Total de instancias cubiertas por las reglas base: {total_instancias}")

            # 2. Calculamos el Soporte (peso real) de cada regla en el modelo
            df_mamdani = df_global_sin_conflictos.copy()
            df_mamdani['Soporte'] = df_mamdani['Frecuencia'] / total_instancias

            # 3. Aplicamos la Poda Mamdani (Filtramos las reglas robustas)
            df_modelo_final = df_mamdani[df_mamdani['Soporte'] >= self.min_soporte].copy()

            # 4. Ordenamos de mayor a menor importancia para facilitar la lectura humana
            df_modelo_final = df_modelo_final.sort_values(by='Frecuencia', ascending=False).reset_index(drop=True)

            # 5. Formateamos visualmente para la tabla de salida
            df_modelo_final['Soporte_Porcentaje'] = (df_modelo_final['Soporte'] * 100).apply(lambda x: f"{x:.2f}%")
            self.modelo_global_difuso = df_modelo_final
            print(f"\n¡Poda completada con éxito!")
            print(f"Reglas originales extraídas de LIME: {len(df_global_sin_conflictos)}")
            print(f"Reglas supervivientes para el motor Mamdani: {len(df_modelo_final)}\n")
        else:
            self.modelo_global_difuso = df_global_sin_conflictos

        print(f"✅ Entrenamiento completado. Modelo Subrogado encapsulado en {len(self.modelo_global_difuso)} reglas maestras.")

    def explain_instance(self, data_row):
        """Motor Mamdani para la explicación local en tiempo real"""
        if self.modelo_global_difuso is None:
             raise ValueError("Debes llamar al método .fit() antes de predecir.")
             
        pertenencias_entrada = {}
        for feature in self.vocabulario_entrada.keys():
            val = data_row[feature]
            vocab = self.vocabulario_entrada[feature]
            universo = vocab['universo']
            val_clipped = np.clip(val, universo[0], universo[-1])
            idx = (np.abs(universo - val_clipped)).argmin()
            pertenencias_entrada[feature] = {etiq: vocab[etiq][idx] for etiq in vocab.keys() if etiq != 'universo'}
                    
        activaciones_salida = {etiq: 0.0 for etiq in self.vocabulario_salida.keys() if etiq != 'universo'}
        reglas_disparadas = []
        
        for _, row in self.modelo_global_difuso.iterrows():
            condicion = row['Condicion']
            pred_ling = row['Prediccion_Linguistica']
            
            grados_condicion = []
            for parte in condicion.split(" Y "):
                var, _, etiq = parte.partition(" es ")
                if var in pertenencias_entrada and etiq in pertenencias_entrada[var]:
                    grados_condicion.append(pertenencias_entrada[var][etiq])
                else:
                    grados_condicion.append(0.0)
                    
            grado_activacion = min(grados_condicion) if grados_condicion else 0.0
            
            if grado_activacion > 0:
                activaciones_salida[pred_ling] = max(activaciones_salida[pred_ling], grado_activacion)
                reglas_disparadas.append({
                    'Fuerza_Activacion': grado_activacion,
                    'Texto_Regla': f"SI {condicion} ENTONCES Prediccion = {pred_ling}"
                })
                
        reglas_disparadas = sorted(reglas_disparadas, key=lambda x: x['Fuerza_Activacion'], reverse=True)
            
        if sum(activaciones_salida.values()) == 0:
            pred_linguistica_oficial = "DESCONOCIDA (Falta cobertura)"
            pred_crisp_oculta = np.mean(self.vocabulario_salida['universo'])
        else:
            pred_linguistica_oficial = max(activaciones_salida, key=activaciones_salida.get)
            universo_out = self.vocabulario_salida['universo']
            agregacion_final = np.zeros_like(universo_out)
            for etiq, activacion in activaciones_salida.items():
                if activacion > 0:
                    agregacion_final = np.fmax(agregacion_final, np.fmin(activacion, self.vocabulario_salida[etiq]))
            pred_crisp_oculta = fuzz.defuzz(universo_out, agregacion_final, 'centroid')
            
        paquete_explicacion = {
            'Prediccion_Linguistica': pred_linguistica_oficial,
            'Prediccion_Numerica': pred_crisp_oculta,
            'Explicaciones': reglas_disparadas
        }
        
        valor_caja_negra = self.predict_fn(data_row.values.reshape(1, -1))[0]
        
        return ExplicacionDifusa(paquete_explicacion, valor_caja_negra)

In [23]:
# 1. El usuario importa tu herramienta
# from fuzzy_lime import FuzzyLimeExplainer

# 2. INICIALIZAR Y ENTRENAR (El usuario no ve las matemáticas, solo invoca la herramienta)
# num_instancias_a_procesar: puedes omitirlo o poner len(X_train) para entrenamiento completo.
explicador = FuzzyLimeExplainer(
    training_data=X_train, 
    predict_fn=rf.predict,
    max_depth=3,            # Soporta cualquier profundidad gracias a la fase 3
    min_soporte=0.02          # Límite máximo de reglas globales legibles
)

# Ejecuta todo el pipeline (Fases 1 a 5) y se guarda el conocimiento internamente
explicador.fit()  

# ----------------------------------------------------------------------------------

# 3. EXPLICAR NUEVAS INSTANCIAS (Se ejecuta al instante)
fila_nueva = X_test.iloc[0]
explicacion = explicador.explain_instance(fila_nueva)

# 4. MOSTRAR RESULTADO
explicacion.show_in_notebook()

Iniciando entrenamiento de Fuzzy-LIME (4000 instancias)...
Total de instancias cubiertas por las reglas base: 28698

¡Poda completada con éxito!
Reglas originales extraídas de LIME: 15
Reglas supervivientes para el motor Mamdani: 11

✅ Entrenamiento completado. Modelo Subrogado encapsulado en 11 reglas maestras.


In [24]:
# 3. EXPLICAR NUEVAS INSTANCIAS (Se ejecuta al instante)
for i in range(5, 10):
    fila_nueva = X_test.iloc[i]
    explicacion = explicador.explain_instance(fila_nueva)

    # 4. MOSTRAR RESULTADO
    explicacion.show_in_notebook()

In [25]:
explicador2 = FuzzyLimeExplainer(
    training_data=X_train, 
    predict_fn=rf.predict,
    max_depth=3,            # Soporta cualquier profundidad gracias a la fase 3
    min_soporte=0        # Sin límite máximo de reglas globales legibles
)

# Ejecuta todo el pipeline (Fases 1 a 5) y se guarda el conocimiento internamente
explicador2.fit()  

# ----------------------------------------------------------------------------------

# 3. EXPLICAR NUEVAS INSTANCIAS (Se ejecuta al instante)
fila_nueva = X_test.iloc[0]
explicacion = explicador2.explain_instance(fila_nueva)

# 4. MOSTRAR RESULTADO
explicacion.show_in_notebook()

Iniciando entrenamiento de Fuzzy-LIME (4000 instancias)...


✅ Entrenamiento completado. Modelo Subrogado encapsulado en 15 reglas maestras.


In [26]:
# 3. EXPLICAR NUEVAS INSTANCIAS (Se ejecuta al instante)
for i in range(5, 10):
    fila_nueva = X_test.iloc[i]
    explicacion2 = explicador2.explain_instance(fila_nueva)

    # 4. MOSTRAR RESULTADO
    explicacion2.show_in_notebook()

# **Fin**